# Chapter 04 제출 답안. pandas로 데이터에 질문하기

## 0. 제출 정보

- 이름: 이성민
- GitHub ID: minilee0307-dot
- 작성·실행일: 2026-09-30
- 최종 제출 URL: https://github.com/minilee0307-dot/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb
- 실습 상태: **COMPLETE** — 전체 코드 셀을 순서대로 실행하고 결과를 저장했다.
- GitHub 표시·eTL 제출 상태: **COMPLETE** — GitHub 본문·출력·Evidence 6장의 표시를 확인했고, 위 Notebook URL로 eTL에 2026-09-30 오후 5:16 제출했다.

### 핵심 답변

**completed 주문 기준 금액이 가장 큰 카테고리는 스포츠다.** 금액은 **31,743,000원**, 수량은 **295개**이며, 전체 완료 주문 금액 148,990,000원의 21.31%를 차지한다. 2위 전자기기와의 차이는 5,343,000원이다.

| 확인 항목 | 이번 분석의 답과 실제 확인값 | 근거 위치 |
| --- | --- | --- |
| 질문·계산 범위 | 2025-09-09~2026-09-09 양 끝 포함, completed만 선택. 상세 금액은 quantity × unit_price | 1번, 실행 셀 [3] |
| 실제 데이터 확인 | CSV 4개. 결측·전체 행 중복·기본키 중복·외래키 미연결 모두 0 | 1번, 실행 셀 [1]~[2] |
| 안전한 병합 | orders 764→764행, products 474→474행. many_to_one 통과, 미연결 0 | 3번, 실행 셀 [6] |
| 네 가지 집계 | 카테고리 7개·상품 98개·월 13개·고객 100명. 완료 주문 184건, 상세 474행 | 4번, 실행 셀 [7]~[9] |
| 총합 교차 검증 | 원본 독립 계산·카테고리·상품·월·고객 합계 모두 148,990,000원, 차이 0원 | 5번, 실행 셀 [10] |
| 저장·LLM 검증 | 결과 CSV 4개 재읽기 일치. 교재 예시의 누락 컬럼과 count 474건 / nunique 184건 차이 확인 | 5~6번, 실행 셀 [11]~[13] |
| 해석의 범위 | 완료 주문 금액의 순위로 답함. 인기도·회계상 순매출·이익 순위는 단정하지 않음 | 4번 해석·한계, 7번 인사이트 |

공식 [실습 Notebook](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/notebooks/ch04_pandas_basic.ipynb), [가이드](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter04/chapter04.md), [제출 양식](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter04/templates/chapter04_assignment.md)과 eTL 과제 안내를 기준으로 작성했다. 예제의 선택·병합·집계 흐름을 사용하고, 실행 결과에 대한 설명과 추가 검증을 붙였다.

이번 작성에는 Codex의 코드 작성·실행·검토·문장 정리 도움을 받았다. 교재의 LLM 검증 예시와 이번에 실제 수행한 검증을 6번에서 구분했다. 데이터는 Chapter 03에서 보존한 **수업용 가상 데이터**와 동일하다. 날짜가 달라지지 않도록 새로 생성하지 않았다.

재실행하려면 저장소를 내려받고 Python 가상환경에 `chapter04/requirements.txt`를 설치한 뒤 그 환경을 커널로 선택하여 전체 실행한다. 원본은 `data/`, 생성되는 요약표는 `reports/`에 있다. 날짜 조건과 원본 파일 해시가 다르면 분석 범위를 다시 확인하도록 했다.

## 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문

**2025-09-09부터 2026-09-09까지(양 끝 날짜 포함), `completed` 주문의 `quantity × unit_price` 합계가 가장 큰 상품 카테고리는 무엇인가?** 같은 범위에서 상품·월·고객별 결과도 확인한다.

### 사용한 파일/컬럼

| 파일 | 주 분석에 사용하는 컬럼 | 역할 |
| --- | --- | --- |
| order_items.csv | order_item_id, order_id, product_id, quantity, unit_price | 상세 행 식별 및 거래 금액 계산 |
| orders.csv | order_id, customer_id, order_date, order_status | 완료 여부, 날짜, 주문과 고객 연결 |
| products.csv | product_id, product_name, category | 상품명과 카테고리 연결 |
| customers.csv | customer_id, city | 고객별 집계에 도시 연결 |

`age`, `gender`, 상품의 `price`, 주문의 `payment_method`는 예제의 필터링 및 추가 실습에서 사용한다. 고객 이름은 출력·집계에 사용하지 않는다. 상품 목록의 `price` 대신 주문 시점의 `unit_price`로 금액을 계산한다.

In [1]:
from pathlib import Path
import sys, hashlib
import pandas as pd
from IPython.display import display

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

def find_chapter(start):
    for parent in [start.resolve(), *start.resolve().parents]:
        for candidate in [parent, parent / 'chapter04']:
            if (candidate / 'chapter04.ipynb').exists() and (candidate / 'data/orders.csv').exists():
                return candidate
    raise FileNotFoundError('chapter04/data의 CSV 4개를 확인하세요.')

CHAPTER = find_chapter(Path.cwd())
DATA_DIR = CHAPTER / 'data'
REPORT_DIR = CHAPTER / 'reports'
REPORT_DIR.mkdir(exist_ok=True)
expected_hashes = {
    'customers.csv': 'cd53b7f52f960a13a67f509230f5a5ab6c68b3cdce2158ff57e6fce8a32917d8',
    'orders.csv': '67cc403001e68134ff2fca73266a8f3b706aca05741c593d30513d4143d1fb7e',
    'products.csv': '46ff06753ade3e71011ee08dd366148dcd9c36521f111840cead0a9d1047e405',
    'order_items.csv': 'a7a5c3942b0886c66cb38cc6a10fdfa68dbcde90e8642585dee49c4bd104063a',
}
file_check = pd.DataFrame([{
    'file': name, 'exists': (DATA_DIR / name).is_file(),
    'sha256_match': hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() == expected
} for name, expected in expected_hashes.items()])
assert file_check[['exists', 'sha256_match']].all().all()
print('Python:', sys.version.split()[0], '/ pandas:', pd.__version__)
print('가상환경:', sys.prefix != sys.base_prefix, '/ 실행 파일:', '/'.join(Path(sys.executable).parts[-3:]))
print('데이터: chapter04/data / 결과: chapter04/reports')
display(file_check)
datasets = {Path(name).stem: pd.read_csv(DATA_DIR / name) for name in expected_hashes}
customers, orders, products, order_items = [datasets[n] for n in ['customers', 'orders', 'products', 'order_items']]
expected_columns = {
    'customers': ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date'],
    'orders': ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'order_items': ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price'],
}
for name, cols in expected_columns.items():
    assert datasets[name].columns.tolist() == cols, f'{name}: 컬럼 확인 필요'
display(pd.DataFrame([{'table': n, 'rows': len(d), 'columns': len(d.columns),
    'missing_cells': int(d.isna().sum().sum()), 'duplicate_rows': int(d.duplicated().sum())}
    for n, d in datasets.items()]))
display(pd.DataFrame([{'table': n, 'column': c, 'dtype_read': str(d[c].dtype)}
    for n, d in datasets.items() for c in d.columns]))
assert all(not d.isna().any().any() for d in datasets.values())

Python: 3.14.3 / pandas: 3.0.5
가상환경: True / 실행 파일: .venv/bin/python
데이터: chapter04/data / 결과: chapter04/reports


,file,exists,sha256_match
0,customers.csv,True,True
1,orders.csv,True,True
2,products.csv,True,True
3,order_items.csv,True,True


,table,rows,columns,missing_cells,duplicate_rows
0,customers,150,6,0,0
1,orders,300,5,0,0
2,products,100,4,0,0
3,order_items,764,5,0,0


,table,column,dtype_read
0,customers,customer_id,int64
1,customers,name,str
2,customers,gender,str
3,customers,age,int64
4,customers,city,str
5,customers,signup_date,str
6,orders,order_id,int64
7,orders,customer_id,int64
8,orders,order_date,str
9,orders,payment_method,str


In [2]:
primary_keys = {'customers': 'customer_id', 'orders': 'order_id',
                'products': 'product_id', 'order_items': 'order_item_id'}
key_check = pd.DataFrame([{'table': n, 'key': k,
    'missing': int(datasets[n][k].isna().sum()),
    'duplicates': int(datasets[n][k].duplicated().sum())} for n, k in primary_keys.items()])
display(key_check)
assert (key_check[['missing', 'duplicates']] == 0).all().all()
fk_check = pd.DataFrame([
    {'relationship': 'orders.customer_id → customers', 'unmatched': int((~orders.customer_id.isin(customers.customer_id)).sum())},
    {'relationship': 'order_items.order_id → orders', 'unmatched': int((~order_items.order_id.isin(orders.order_id)).sum())},
    {'relationship': 'order_items.product_id → products', 'unmatched': int((~order_items.product_id.isin(products.product_id)).sum())},
])
display(fk_check)
assert fk_check.unmatched.eq(0).all()
numeric_ranges = pd.DataFrame([{'column': f'{n}.{c}', 'min': int(datasets[n][c].min()),
    'max': int(datasets[n][c].max())} for n, c in [('customers', 'age'), ('products', 'price'),
    ('order_items', 'quantity'), ('order_items', 'unit_price')]])
display(numeric_ranges)
assert order_items[['quantity', 'unit_price']].gt(0).all().all()
print('주문 상태별 주문 건수')
display(orders.order_status.value_counts(dropna=False).rename_axis('order_status').to_frame('orders'))
print('카테고리별 등록 상품 수')
display(products.category.value_counts(dropna=False).rename_axis('category').to_frame('products'))

,table,key,missing,duplicates
0,customers,customer_id,0,0
1,orders,order_id,0,0
2,products,product_id,0,0
3,order_items,order_item_id,0,0


,relationship,unmatched
0,orders.customer_id → customers,0
1,order_items.order_id → orders,0
2,order_items.product_id → products,0


,column,min,max
0,customers.age,19,69
1,products.price,5000,200000
2,order_items.quantity,1,5
3,order_items.unit_price,5000,200000


주문 상태별 주문 건수


,orders
order_status,
completed,184
cancelled,64
refunded,52


카테고리별 등록 상품 수


,products
category,
스포츠,19
전자기기,17
생활용품,16
뷰티,16
도서,14
패션,11
식품,7


In [3]:
START_DATE = pd.Timestamp('2025-09-09')
END_DATE = pd.Timestamp('2026-09-09')
parsed_dates = pd.to_datetime(orders.order_date, format='%Y-%m-%d', errors='coerce')
assert parsed_dates.notna().all(), '날짜 변환 실패를 먼저 확인하세요.'
assert parsed_dates.between(START_DATE, END_DATE).all(), '원본의 기간이 분석 범위와 달라졌습니다.'
scope_table = pd.DataFrame([
    ['질문', 'completed 주문 금액이 가장 큰 카테고리는?'],
    ['분석 기간', f'{START_DATE.date()} ~ {END_DATE.date()} (양 끝 포함)'],
    ['실제 주문일 범위', f'{parsed_dates.min().date()} ~ {parsed_dates.max().date()}'],
    ['날짜 변환 실패', str(int(parsed_dates.isna().sum()))],
    ['상태', 'completed만 포함 / cancelled·refunded 제외'],
    ['상세 행 금액', 'line_total = quantity × unit_price'],
    ['집계', 'total_sales = 완료 주문 상세 line_total의 합계'],
    ['주문 수', 'order_count = order_id.nunique()'],
    ['금액 해석', '샘플 금액 단위를 원으로 표시. 회계상 순매출·이익 아님'],
], columns=['항목', '기준'])
display(scope_table)

,항목,기준
0,질문,completed 주문 금액이 가장 큰 카테고리는?
1,분석 기간,2025-09-09 ~ 2026-09-09 (양 끝 포함)
2,실제 주문일 범위,2025-09-09 ~ 2026-09-09
3,날짜 변환 실패,0
4,상태,completed만 포함 / cancelled·refunded 제외
5,상세 행 금액,line_total = quantity × unit_price
6,집계,total_sales = 완료 주문 상세 line_total의 합계
7,주문 수,order_count = order_id.nunique()
8,금액 해석,샘플 금액 단위를 원으로 표시. 회계상 순매출·이익 아님


![질문과 계산 기준](images/step01_question.png)

### 결과 관찰
네 CSV는 customers 150×6, orders 300×5, products 100×4, order_items 764×5이다. 결측값과 전체 행 중복은 모두 0이고, 각 기본키의 결측·중복 및 세 외래키의 미연결도 0이다. ID·나이·가격·수량·단가는 `int64`, 텍스트와 읽기 직후 날짜는 `str`이다. 실제 주문일 범위는 2025-09-09~2026-09-09이고 변환 실패는 0건이다. 상태는 completed 184건, cancelled 64건, refunded 52건이다.

### 나의 해석과 판단
이번 질문은 고객 나이나 도시로 대상을 제한하지 않는다. 원본의 전체 기간에서 완료 주문만 비교하기로 했다. 주문 상세의 수량과 단가만으로는 카테고리나 완료 여부를 알 수 없으므로 orders와 products를 연결해야 한다.

### 업무·분석적 의미
금액의 범위와 계산식을 먼저 정하면 이후 표들이 같은 기준으로 비교된다. 주문 상세의 행 수와 실제 주문 수를 구분하는 것도 중복 집계를 막는 데 필요하다.

### 한계와 추가 확인 사항
수업용 가상 데이터의 결과이므로 실제 소비자 수요로 일반화하지 않는다. ID가 정상 연결된다는 사실은 거래 내용이 현실적으로 타당하다는 뜻까지 보장하지 않는다. 특히 양 끝의 2025년 9월과 2026년 9월은 한 달 전체가 아니다.

## 2. 필터링·정렬·파생 컬럼

- 연습 필터: `age >= 30`, `city == '서울'`, `city.isin(['서울', '부산'])`. 주 분석에는 적용하지 않는다.
- 주 분석 필터: 정해진 기간의 `order_status == 'completed'`.
- 정렬: 상품 `price`와 카테고리·상품·고객의 `total_sales`는 내림차순이다. 동률은 ID나 카테고리 이름순으로 정렬한다. **월별 표는 시간 흐름을 보기 위해 `order_month` 오름차순**으로 정렬한다.
- 파생 컬럼: `line_total = quantity * unit_price`, `order_month = order_date.dt.to_period('M')`.

In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customers.city.value_counts(dropna=False).rename_axis('city').to_frame('customers'))
customers_over_30 = customer_basic[customer_basic.age >= 30]
seoul_customers = customer_basic[customer_basic.city == '서울']
city_customers = customer_basic[customer_basic.city.isin(['서울', '부산'])]
display(pd.DataFrame({'조건': ['30세 이상', '서울', '서울 또는 부산'],
    '고객 수': [len(customers_over_30), len(seoul_customers), len(city_customers)]}))
display(products.sort_values(['price', 'product_id'], ascending=[False, True]).head(10))

,customers
city,
성남,21
광주,17
부산,16
대구,15
서울,15
울산,14
인천,14
대전,14
수원,13


,조건,고객 수
0,30세 이상,111
1,서울,15
2,서울 또는 부산,31


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
42,43,뷰티 상품 043,뷰티,197000
57,58,식품 상품 058,식품,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
7,8,스포츠 상품 008,스포츠,189000
71,72,뷰티 상품 072,뷰티,189000
52,53,생활용품 상품 053,생활용품,188000


In [5]:
order_items = order_items.copy()
order_items['line_total'] = order_items.quantity * order_items.unit_price
all_order_amount = int(order_items.line_total.sum())
print('line_total = quantity × unit_price')
print('전체 상태 주문 상세:', len(order_items), '행')
print(f'전체 상태의 주문 상세 금액: {all_order_amount:,}원 (completed 금액 아님)')
display(order_items[['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())

line_total = quantity × unit_price
전체 상태 주문 상세: 764 행
전체 상태의 주문 상세 금액: 255,610,000원 (completed 금액 아님)


,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰
`quantity`는 1~5, `unit_price`와 상품 `price`는 5,000~200,000이다. 상세 764행에 `line_total`을 만들었으며 상태를 구분하기 전 합계는 255,610,000원이다. 30세 이상 고객은 111명, 서울은 15명, 서울 또는 부산은 31명이다. 가격 내림차순 1위는 뷰티 상품 099의 200,000원이다.

### 나의 해석과 판단
전체 상태 합계에는 취소·환불 주문도 들어 있다. 이 금액을 완료 주문 금액으로 쓰면 범위가 달라진다. 서울 고객이나 특정 연령만 선택해도 모집단이 달라지므로 연습용 필터를 주 분석에 섞지 않았다.

### 업무·분석적 의미
가격 정렬은 고가 상품을 확인하는 데 쓸 수 있지만 판매 금액 순위와 같지는 않다. 판매 수량과 실제 주문 단가가 함께 필요하다.

### 한계와 추가 확인 사항
수량과 단가의 곱에는 별도 할인, 배송비, 세금, 원가를 반영하지 않았다. 컬럼이 없으므로 임의로 추정하여 빼거나 더하지 않았다.

## 3. merge 검증

주문 상세를 기준으로 왼쪽 병합한다. orders의 `order_id`와 products의 `product_id`가 각각 고유해야 한다. `many_to_one` 검증과 `indicator=True`를 함께 사용하고, 다음 병합 전에는 `_merge`를 제거한다. 행 수와 상세 행 ID가 보존되는지도 확인한다.

In [6]:
order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id', how='left', validate='many_to_one', indicator=True)
merge_rows = [{'merge': 'order_items + orders', 'validate': 'many_to_one',
    'before': len(order_items), 'after': len(order_sales),
    **{k: int(order_sales._merge.eq(k).sum()) for k in ['both', 'left_only', 'right_only']}}]
assert len(order_sales) == len(order_items)
assert order_sales._merge.eq('both').all()
assert order_sales.order_item_id.is_unique
order_sales = order_sales.drop(columns='_merge')
order_sales['order_date'] = pd.to_datetime(order_sales.order_date, format='%Y-%m-%d', errors='coerce')
assert order_sales.order_date.notna().all()
completed_order_sales = order_sales.loc[
    order_sales.order_status.eq('completed') & order_sales.order_date.between(START_DATE, END_DATE)
].copy()
assert set(completed_order_sales.order_status) == {'completed'}
completed_order_sales['order_month'] = completed_order_sales.order_date.dt.to_period('M').astype(str)

completed_sales_items = completed_order_sales.merge(
    products[['product_id', 'product_name', 'category']],
    on='product_id', how='left', validate='many_to_one', indicator=True)
merge_rows.append({'merge': 'completed + products', 'validate': 'many_to_one',
    'before': len(completed_order_sales), 'after': len(completed_sales_items),
    **{k: int(completed_sales_items._merge.eq(k).sum()) for k in ['both', 'left_only', 'right_only']}})
assert len(completed_sales_items) == len(completed_order_sales)
assert completed_sales_items._merge.eq('both').all()
assert completed_sales_items.order_item_id.is_unique
assert completed_sales_items.category.notna().all()
completed_sales_items = completed_sales_items.drop(columns='_merge')
merge_check = pd.DataFrame(merge_rows)
print('orders.order_id 중복:', int(orders.order_id.duplicated().sum()))
print('products.product_id 중복:', int(products.product_id.duplicated().sum()))
display(merge_check)
print('날짜 변환 실패(상세 행):', int(order_sales.order_date.isna().sum()))
print('원본 주문 중 상세 없는 주문:', int((~orders.order_id.isin(order_items.order_id)).sum()))
assert orders.order_id.isin(order_items.order_id).all()

orders.order_id 중복: 0
products.product_id 중복: 0


,merge,validate,before,after,both,left_only,right_only
0,order_items + orders,many_to_one,764,764,764,0,0
1,completed + products,many_to_one,474,474,474,0,0


날짜 변환 실패(상세 행): 0
원본 주문 중 상세 없는 주문: 0


![merge 검증](images/step03_merge.png)

### 결과 관찰
orders의 주문 ID와 products의 상품 ID 중복은 모두 0이다. 첫 병합은 764→764행, 상품 병합은 완료 주문 상세 474→474행으로 유지됐다. 두 병합 모두 `many_to_one` 검증을 통과했고, `both`는 각각 764·474행, `left_only`와 `right_only`는 0행이다. 원본 주문 300건 중 상세가 없는 주문도 0건이다.

### 나의 해석과 판단
오른쪽 키가 고유하고 모든 왼쪽 상세가 연결됐으며 상세 ID도 고유하게 유지되어, 이번 병합에서 상세가 누락되거나 복제된 문제는 발견되지 않았다. 764행이 474행으로 줄어든 것은 병합 손실이 아니라 완료 상태 필터 때문이다. 왼쪽 병합에서 `right_only=0`만으로 모든 오른쪽 행이 사용됐다고 판단할 수는 없다.

### 업무·분석적 의미
오른쪽 키 중복이 있으면 같은 상세가 여러 번 붙어 금액이 커질 수 있다. 반대로 미연결 카테고리를 무시하고 집계하면 일부 금액이 빠질 수 있다. 실행 성공과 별도로 두 경우를 확인해야 한다.

### 한계와 추가 확인 사항
이번 검증은 연결 구조에 대한 것이다. 원본에서 같은 주문·상품 조합이 반복되더라도 상세 ID가 다르면 근거 없이 삭제하지 않았다. 실제 거래라면 분할 주문인지 입력 오류인지는 추가 업무 규칙이 필요하다.

## 4. completed 주문 범위와 집계

상태별 비교표는 필터 효과를 설명하기 위한 것이며, 아래 카테고리·상품·월·고객 요약표 네 개에는 모두 **completed 184건의 상세 474행**만 사용한다. 주문 수는 상세 행 개수가 아니라 `order_id.nunique()`로 계산한다.

분석 단위는 상세 1행, 주문 1건, 카테고리 1개를 구분한다. 상세의 수량×단가를 더한 값이 금액이고, 주문 ID의 고유 개수가 주문 수다. 네 표가 집계하는 전체 금액은 동일한 148,990,000원이며, 카테고리 금액 비중은 각 금액을 이 전체 합계로 나눈 뒤 100을 곱한다. 아래 13개 월은 **13개월의 온전한 관측 기간을 뜻하지 않는다.** 분석 기간 안에 걸친 달의 개수이며, 두 끝의 9월은 일부 기간만 포함한다.


In [7]:
status_summary = order_sales.groupby('order_status', as_index=False).agg(
    detail_rows=('order_item_id', 'size'), order_count=('order_id', 'nunique'),
    total_quantity=('quantity', 'sum'), total_amount=('line_total', 'sum'))
display(status_summary)
print('완료 주문 실제 날짜:', completed_order_sales.order_date.min().date(), '~', completed_order_sales.order_date.max().date())
category_sales = completed_sales_items.groupby('category', as_index=False, dropna=False).agg(
    total_quantity=('quantity', 'sum'), total_sales=('line_total', 'sum'))
category_sales = category_sales.sort_values(['total_sales', 'category'], ascending=[False, True]).reset_index(drop=True)
category_sales['sales_ratio'] = (category_sales.total_sales / category_sales.total_sales.sum() * 100).round(2)
print('카테고리별 완료 주문 금액 (원), 수량 (개), 비중 (%)')
display(category_sales)

,order_status,detail_rows,order_count,total_quantity,total_amount
0,cancelled,162,64,507,62181000
1,completed,474,184,1442,148990000
2,refunded,128,52,384,44439000


완료 주문 실제 날짜: 2025-09-22 ~ 2026-09-09
카테고리별 완료 주문 금액 (원), 수량 (개), 비중 (%)


,category,total_quantity,total_sales,sales_ratio
0,스포츠,295,31743000,21.31
1,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
3,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
5,도서,149,16389000,11.00
6,패션,111,10587000,7.11


In [8]:
product_sales = completed_sales_items.groupby(
    ['product_id', 'product_name', 'category'], as_index=False, dropna=False).agg(
    total_quantity=('quantity', 'sum'), total_sales=('line_total', 'sum'))
product_sales = product_sales.sort_values(['total_sales', 'product_id'], ascending=[False, True]).reset_index(drop=True)
print('완료 주문이 있는 상품:', len(product_sales), '/ 등록 상품:', len(products))
print('상품별 완료 주문 금액 상위 10개 (전체 결과는 reports CSV)')
display(product_sales.head(10))

monthly_summary = completed_order_sales.groupby('order_month', as_index=False, dropna=False).agg(
    total_sales=('line_total', 'sum'), order_count=('order_id', 'nunique'))
monthly_summary = monthly_summary.sort_values('order_month').reset_index(drop=True)
monthly_summary['average_order_value'] = (monthly_summary.total_sales / monthly_summary.order_count).round(0)
print('월별 완료 주문 금액·고유 주문 수·주문당 평균 금액 (원)')
display(monthly_summary)

완료 주문이 있는 상품: 98 / 등록 상품: 100
상품별 완료 주문 금액 상위 10개 (전체 결과는 reports CSV)


,product_id,product_name,category,total_quantity,total_sales
0,41,스포츠 상품 041,스포츠,35,5705000
1,12,식품 상품 012,식품,25,4375000
2,9,스포츠 상품 009,스포츠,20,3860000
3,72,뷰티 상품 072,뷰티,20,3780000
4,71,전자기기 상품 071,전자기기,23,3703000
5,68,스포츠 상품 068,스포츠,26,3640000
6,81,전자기기 상품 081,전자기기,22,3630000
7,11,패션 상품 011,패션,31,3565000
8,22,생활용품 상품 022,생활용품,29,3248000
9,89,생활용품 상품 089,생활용품,30,3090000


월별 완료 주문 금액·고유 주문 수·주문당 평균 금액 (원)


,order_month,total_sales,order_count,average_order_value
0,2025-09,7932000,8,"991,500.00"
1,2025-10,12482000,14,"891,571.00"
2,2025-11,13104000,18,"728,000.00"
3,2025-12,10665000,13,"820,385.00"
4,2026-01,5000000,8,"625,000.00"
5,2026-02,12187000,15,"812,467.00"
6,2026-03,17577000,21,"837,000.00"
7,2026-04,15848000,19,"834,105.00"
8,2026-05,10606000,14,"757,571.00"
9,2026-06,8563000,10,"856,300.00"


In [9]:
customer_sales = completed_order_sales.groupby('customer_id', as_index=False, dropna=False).agg(
    order_count=('order_id', 'nunique'), total_sales=('line_total', 'sum'))
customer_sales = customer_sales.sort_values(['total_sales', 'customer_id'], ascending=[False, True]).reset_index(drop=True)
before_customers = len(customer_sales)
customer_sales = customer_sales.merge(customers[['customer_id', 'city']],
    on='customer_id', how='left', validate='one_to_one', indicator=True)
print('고객 속성 병합: one_to_one / 전후 행 수:', before_customers, len(customer_sales))
display(customer_sales._merge.value_counts().rename_axis('indicator').to_frame('rows'))
assert before_customers == len(customer_sales)
assert customer_sales._merge.eq('both').all()
customer_sales = customer_sales.drop(columns='_merge')
customer_sales['customer_label'] = 'Customer ' + customer_sales.customer_id.astype(str)
customer_sales = customer_sales[['customer_id', 'customer_label', 'city', 'order_count', 'total_sales']]
print('완료 주문이 있는 고객:', len(customer_sales), '/ 전체 고객:', len(customers))
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))

고객 속성 병합: one_to_one / 전후 행 수: 100 100


,rows
indicator,
both,100
left_only,0
right_only,0


완료 주문이 있는 고객: 100 / 전체 고객: 150


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰

- **카테고리:** 스포츠가 31,743,000원, 295개로 금액 1위이며 전체 148,990,000원의 21.31%다. 전자기기 26,400,000원, 생활용품 23,915,000원이 뒤를 이었다. 패션은 10,587,000원으로 가장 작았다.
- **상품:** 완료 주문이 있는 상품은 98개다. 금액 1위는 스포츠 상품 041로 35개, 5,705,000원이다. 등록 상품 100개 중 완료 주문이 없는 2개는 이 집계에 행이 없다.
- **월:** 13개 월 중 2026년 3월이 17,577,000원, 21건으로 가장 크다. 2026년 1월은 5,000,000원, 8건이다. 필터를 통과한 완료 주문의 실제 날짜는 2025-09-22~2026-09-09다.
- **고객:** 완료 주문이 있는 고객은 100명이다. Customer 117이 5건, 4,100,000원으로 금액 1위다. 고객 속성 병합은 one_to_one을 통과했고 100→100행, both 100행, 미연결 0행이었다. 나머지 50명은 완료 주문이 없어 표에 포함되지 않는다.

### 나의 해석과 판단
질문의 답은 스포츠다. 다만 금액 순위와 수량 순위는 구분해야 한다. 생활용품은 272개로 전자기기의 259개보다 많이 팔렸지만, 금액은 전자기기보다 2,485,000원 작다. 금액이 더 큰 이유를 수요가 많아서라고만 설명하기보다 수량과 단가를 함께 살펴봐야 한다고 판단했다.

### 업무·분석적 의미
스포츠는 현재 완료 주문 금액의 기여가 가장 큰 카테고리이므로 상품별 구성을 먼저 확인할 대상으로 삼을 수 있다. 생활용품과 전자기기는 수량과 금액의 순위가 다르므로 재고 판단에는 수량도 같이 보는 편이 낫다. 월별로는 3월의 높은 금액이 주문 수와 주문당 금액 중 어느 쪽과 더 관련되는지 나눠 볼 수 있다.

### 한계와 추가 확인 사항
금액에는 할인·세금·배송비·원가·별도 환불 조정이 반영되지 않아 회계상 순매출이나 이익이라고 할 수 없다. 스포츠가 실제로 가장 인기 있거나 수익성이 높다는 결론도 내리지 않았다. 월별로 양 끝 9월은 일부 날짜만 있으므로 전체 월과 단순 비교하여 성장·하락을 단정하지 않았다. 완료 주문이 없는 상품·고객을 0으로 보여 주는 표가 필요하면 전체 목록에 다시 연결해야 한다.

## 5. 총합 일치 검증

병합한 표에서 만든 합계끼리만 비교하면 동일한 오류를 공유할 수 있다. 그래서 원본 orders에서 completed 주문 ID를 얻고, 원본 order_items를 `isin()`으로 선택하여 `quantity × unit_price`를 다시 계산한 값을 기준으로 삼았다. 상품 집계도 빠뜨리지 않고 함께 비교한다.

In [10]:
reference_ids = orders.loc[orders.order_status.eq('completed') &
    parsed_dates.between(START_DATE, END_DATE), 'order_id']
reference_items = datasets['order_items'].loc[datasets['order_items'].order_id.isin(reference_ids)]
reference_total = int((reference_items.quantity * reference_items.unit_price).sum())
totals = {
    '원본 completed (isin으로 독립 계산)': reference_total,
    '완료 주문 상세': int(completed_order_sales.line_total.sum()),
    '카테고리별': int(category_sales.total_sales.sum()),
    '상품별': int(product_sales.total_sales.sum()),
    '월별': int(monthly_summary.total_sales.sum()),
    '고객별': int(customer_sales.total_sales.sum()),
}
total_check = pd.DataFrame([{'기준': k, '합계(원)': v, '원본과 차이(원)': v-reference_total,
    '결과': 'PASS' if v == reference_total else 'FAIL'} for k, v in totals.items()])
display(total_check)
assert total_check['원본과 차이(원)'].eq(0).all()
assert set(reference_items.order_item_id) == set(completed_sales_items.order_item_id)
order_check = pd.DataFrame({'기준': ['원본 completed 주문', '완료 상세의 고유 주문', '월별 주문 수 합', '고객별 주문 수 합'],
    '주문 수': [len(reference_ids), completed_order_sales.order_id.nunique(),
                int(monthly_summary.order_count.sum()), int(customer_sales.order_count.sum())]})
display(order_check)
assert order_check['주문 수'].eq(len(reference_ids)).all()
print('원본·집계 합계 및 상세 ID 보존: PASS')

,기준,합계(원),원본과 차이(원),결과
0,원본 completed (isin으로 독립 계산),148990000,0,PASS
1,완료 주문 상세,148990000,0,PASS
2,카테고리별,148990000,0,PASS
3,상품별,148990000,0,PASS
4,월별,148990000,0,PASS
5,고객별,148990000,0,PASS


,기준,주문 수
0,원본 completed 주문,184
1,완료 상세의 고유 주문,184
2,월별 주문 수 합,184
3,고객별 주문 수 합,184


원본·집계 합계 및 상세 ID 보존: PASS


![총합 검증](images/step05_total_check.png)

### 결과 관찰
원본 completed 독립 계산, 완료 주문 상세, 카테고리, 상품, 월, 고객의 금액은 모두 **148,990,000원**이다. 원본 대비 차이는 모두 0원이다. 원본 주문 수, 상세의 고유 주문 수, 월별 주문 수 합, 고객별 주문 수 합도 모두 184건이다.

### 나의 해석과 판단
이번에는 합계뿐 아니라 원본과 병합 결과의 상세 ID 집합도 일치했다. 합계가 같아도 서로 다른 행의 누락·중복이 상쇄될 수 있으므로 합계만으로 모든 오류가 없다고 보지는 않는다. 불일치하면 ① completed·기간 필터 ② 오른쪽 키 중복과 병합 전후 행 수 ③ 미연결 및 그룹 키 결측 ④ 날짜·숫자 변환 ⑤ 계산식과 반올림 순서로 확인하겠다.

### 업무·분석적 의미
집계 기준이 바뀌어도 같은 거래 범위를 보존했는지 확인하는 장치다. 월·고객별 주문 수는 각 주문이 한 그룹에만 들어가서 합산 가능하지만, 여러 카테고리를 산 주문은 카테고리별 주문 수에서 중복될 수 있다.

### 한계
합계 일치는 원본 데이터 자체가 정확한 거래 기록인지를 증명하지 않는다. 동일한 범위·계산식·키 관계가 맞다는 조건 아래 집계 과정의 일관성을 확인한 결과로 해석했다.

### 결과 CSV 저장과 다시 읽기
네 집계표를 저장한 뒤 모두 다시 읽어 행·열·값을 비교한다. 아래 결과는 화면에 일부만 표시한 상품·고객 표의 전체 결과도 포함한다.

In [11]:
outputs = {'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales}
save_rows = []
for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    reread = pd.read_csv(path)
    pd.testing.assert_frame_equal(df.reset_index(drop=True), reread, check_dtype=False)
    assert int(reread.total_sales.sum()) == reference_total
    save_rows.append({'file': filename, 'exists': path.exists(), 'bytes': path.stat().st_size,
        'rows': len(reread), 'columns': len(reread.columns), 'reread': 'PASS'})
display(pd.DataFrame(save_rows))

,file,exists,bytes,rows,columns,reread
0,ch04_category_sales.csv,True,246,7,4,PASS
1,ch04_product_sales.csv,True,4310,98,5,PASS
2,ch04_monthly_sales.csv,True,430,13,4,PASS
3,ch04_customer_sales.csv,True,3286,100,5,PASS


## 6. LLM pandas 코드 검증

### Prompt Log와 제안 코드 요약

- 도구·날짜: Codex, 2026-09-30.
- 이번 실제 요청 요약: eTL 과제 4와 공식 자료를 확인하여 실제 데이터를 실행하고, 누락된 답변과 검증을 보완한 Notebook을 작성하도록 요청했다. 본문의 코드는 이 과정에서 Codex가 작성·실행한 코드다.
- Codex 제안 요약: 실제 컬럼과 키를 확인하고, 상세 금액을 계산한 뒤 orders·products를 왼쪽 병합한다. completed와 기간 조건을 적용해 네 집계를 만들고, 주문 수에는 nunique를 사용하며 원본 기준 합계와 비교한다.
- 실제 데이터와의 대조: orders.order_id와 products.product_id 중복은 0이고, 병합은 각각 764→764행과 474→474행이다. completed 184건의 상세 474행만 사용하며, 네 집계의 금액이 원본 계산 148,990,000원과 일치한다. 검사한 범위에서 이번 최종 제안 코드의 컬럼·범위 오류는 발견되지 않았다.

### 검토 대상과 최종 판단

| 검토 대상 | 발견한 문제 또는 확인 근거 | 판단 |
| --- | --- | --- |
| 교재가 제시한 검토용 한 줄 코드 | 원본 order_items에 category가 없고, 완료 상태도 알 수 없다. line_total 생성도 선행되어야 한다 | **수정 후 사용** |
| 이번 Codex의 최종 병합·집계 코드 | 실제 컬럼 사용, many_to_one 검증, 완료 상태·기간 필터, 고유 주문 수 및 총합 일치 확인 | **검증 후 사용** |
| 실행 비교용 대안: count 사용 또는 completed 필터 생략 | 주문 수 474건과 금액 255,610,000원으로 분석 질문의 기준과 달라진다 | **사용하지 않음** |
| 금액 1위에서 인기도·이익 1위까지 확대하는 해석 | 고객 선호·원가·할인·별도 환불 조정 정보가 충분하지 않다 | **판단 보류** |

교재의 검토용 코드는 공식 Notebook 14번의 `order_items.groupby('category')['line_total'].sum()`이다. 이를 검토하면서 line_total 생성, orders 병합, 날짜·completed 필터, products 병합, 카테고리 집계와 원본 합계 검증을 추가했다. 수정된 실행 코드는 본문 2~5번에 있다.

### 실행으로 대조한 내용
아래 셀은 교재 예시의 KeyError를 확인하고, 주문 수와 상태 필터를 잘못 선택했을 때의 결과를 비교한다. 표 마지막의 **수정 후 사용**은 교재 예시를 수정한 판단이다. count 사용과 필터 생략은 차이를 확인하기 위한 비교용 계산이며, 별도 LLM 대화에서 실제로 받은 답변으로 기록하지 않았다.


In [12]:
try:
    order_items.groupby('category')['line_total'].sum()
except KeyError as error:
    example_error = f'{type(error).__name__}: {error}'
else:
    raise AssertionError('교재 예시의 전제가 바뀌었습니다. 컬럼을 다시 확인하세요.')

wrong_order_count = int(completed_order_sales.order_id.count())
correct_order_count = int(completed_order_sales.order_id.nunique())
llm_validation = pd.DataFrame([
    ['교재 한 줄 예시', example_error, '수정: category를 products에서 연결'],
    ['금액 범위', f'전체 {all_order_amount:,} / completed {reference_total:,}', '수정: 완료 상태 필터 필요'],
    ['주문 수', f'count {wrong_order_count} / nunique {correct_order_count}', '수정: nunique 사용'],
    ['orders 병합', f'{merge_check.iloc[0]["before"]} → {merge_check.iloc[0]["after"]}, 미연결 0', '사용: many_to_one + indicator'],
    ['products 병합', f'{merge_check.iloc[1]["before"]} → {merge_check.iloc[1]["after"]}, 미연결 0', '사용: many_to_one + indicator'],
    ['총합 대조', f'모든 집계 {reference_total:,}원 / 차이 0원', '사용: 원본과 일치'],
    ['이익·순매출 결론', '원가·할인·배송비·세금 정보 없음', '보류: 현재 자료로 계산 불가'],
], columns=['검토 항목', '실제 확인', '판단'])
display(llm_validation)
print('최종 판단: 수정 후 사용')

,검토 항목,실제 확인,판단
0,교재 한 줄 예시,KeyError: 'category',수정: category를 products에서 연결
1,금액 범위,"전체 255,610,000 / completed 148,990,000",수정: 완료 상태 필터 필요
2,주문 수,count 474 / nunique 184,수정: nunique 사용
3,orders 병합,"764 → 764, 미연결 0",사용: many_to_one + indicator
4,products 병합,"474 → 474, 미연결 0",사용: many_to_one + indicator
5,총합 대조,"모든 집계 148,990,000원 / 차이 0원",사용: 원본과 일치
6,이익·순매출 결론,원가·할인·배송비·세금 정보 없음,보류: 현재 자료로 계산 불가


최종 판단: 수정 후 사용


![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
category가 없는 코드는 오류가 나서 바로 알 수 있었다. 더 주의할 부분은 오류 없이 실행되는 잘못된 계산이다. `count()`는 주문을 474건으로 세지만 실제 고유 주문은 184건이다. completed 필터가 없으면 금액도 106,620,000원 커진다. 그래서 실행 여부에 더해 컬럼 위치, 병합 관계, 분석 범위, 집계 단위와 총합을 확인했다. 금액 1위에서 수익성 1위까지 결론을 넓히는 해석은 자료가 부족해 보류한다.

### 검증 장치가 오류를 실제로 잡는지 확인
원본을 바꾸지 않고 복사본의 orders에 중복 키를 하나 넣어 `validate`가 오류를 내는지 확인한다. 별도 복사본의 주문 ID 하나를 존재하지 않는 값으로 바꾸어 `left_only`가 검출되는지도 확인한다.

In [13]:
duplicate_orders = pd.concat([orders, orders.iloc[[0]]], ignore_index=True)
try:
    order_items.merge(duplicate_orders[['order_id']], on='order_id', how='left', validate='many_to_one')
except pd.errors.MergeError:
    print('중복 오른쪽 키 복사본: MergeError 검출 PASS')
else:
    raise AssertionError('중복 키 검증 실패')
unmatched_sample = order_items.head(1).copy()
unmatched_sample.loc[:, 'order_id'] = int(orders.order_id.max()) + 1
probe = unmatched_sample.merge(orders[['order_id']], on='order_id', how='left',
                               validate='many_to_one', indicator=True)
assert probe._merge.eq('left_only').sum() == 1
print('존재하지 않는 주문 ID 복사본: left_only 1행 검출 PASS')

중복 오른쪽 키 복사본: MergeError 검출 PASS
존재하지 않는 주문 ID 복사본: left_only 1행 검출 PASS


## 7. Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

1. **완료 주문 금액이 가장 큰 카테고리는 스포츠다.** 31,743,000원으로 전체 148,990,000원의 21.31%를 차지한다. 2위 전자기기보다 5,343,000원 크다. 이 결론은 완료 주문 184건, 상세 474행의 집계이며, 네 집계의 총액도 원본과 일치했다.
2. **판매 수량만으로 금액 순위를 설명할 수 없다.** 생활용품은 272개로 수량 2위지만 금액은 23,915,000원으로 3위다. 전자기기는 259개로 수량 3위, 26,400,000원으로 금액 2위다. 카테고리 비교에는 거래 단가와 상품 구성을 함께 봐야 한다.

### 그 결과를 뒷받침하는 수치/표
4번의 카테고리 전체 표, 상품 상위 10개 표와 5번의 총합 검증 표가 근거다. 전체 상품·고객 결과는 reports/의 CSV에 저장했다.

### 추가로 확인하고 싶은 질문
전자기기와 생활용품의 금액 차이는 판매된 상품의 단가 분포와 상품별 수량 구성으로 얼마나 설명되는가? 다음 분석에서는 카테고리별 total_sales / total_quantity와 상품별 기여도를 계산해 비교하고 싶다. 실제 재고 결정을 하려면 재고·원가·기간별 판매 가능일도 추가로 필요하다.

### 현재 결과의 한계
합계 검증은 같은 계산 범위를 유지했는지 확인할 뿐, 가상 데이터가 현실의 소비를 대표한다는 뜻은 아니다. 완료 필터를 생략하면 255,610,000원이 되어 현재 기준보다 106,620,000원(71.56%) 커진다는 점도 확인했다. 분석 기준을 바꾸면 결론을 같은 이름으로 비교하지 않아야 한다.

## 추가 실습. 공식 Notebook 15번의 6문항

### 1~4. 고객 필터, 저가 상품, 결제수단, 카테고리 평균 가격
40세 이상은 customers에서 추출한다. 결제수단별 비율의 분모는 **전체 상태의 orders 300건**이다. 카테고리 평균 가격은 products의 등록 상품 단순 평균으로, 완료 주문에서의 판매 가중 평균과 구분한다.

In [14]:
customers_over_40 = customers.loc[customers.age >= 40, ['customer_id', 'gender', 'age', 'city']]
print('1. 40세 이상 고객:', len(customers_over_40), '명 (앞 10행 표시)')
display(customers_over_40.head(10))
print('2. 가격이 낮은 상품 10개 (동률은 product_id 순)')
display(products.sort_values(['price', 'product_id']).head(10))
print('3. 전체 주문의 결제수단별 건수와 비율')
payment_summary = orders.payment_method.value_counts(dropna=False).rename_axis('payment_method').reset_index(name='order_count')
payment_summary['ratio_pct'] = (payment_summary.order_count / len(orders) * 100).round(2)
assert payment_summary.order_count.sum() == len(orders)
display(payment_summary)
print('4. 카테고리별 등록 상품 평균 가격')
average_prices = products.groupby('category', as_index=False).agg(
    product_count=('product_id', 'nunique'), average_price=('price', 'mean'))
display(average_prices.sort_values(['average_price', 'category'], ascending=[False, True]))

1. 40세 이상 고객: 78 명 (앞 10행 표시)


,customer_id,gender,age,city
2,3,F,61,성남
3,4,F,55,울산
6,7,F,53,인천
8,9,M,69,서울
9,10,F,62,울산
11,12,F,66,대전
13,14,F,40,대전
14,15,M,69,서울
15,16,M,52,부산
17,18,M,58,고양


2. 가격이 낮은 상품 10개 (동률은 product_id 순)


,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


3. 전체 주문의 결제수단별 건수와 비율


,payment_method,order_count,ratio_pct
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


4. 카테고리별 등록 상품 평균 가격


,category,product_count,average_price
4,식품,7,"137,142.86"
1,뷰티,16,"117,687.50"
6,패션,11,"115,909.09"
3,스포츠,19,"111,578.95"
0,도서,14,"106,857.14"
5,전자기기,17,"101,588.24"
2,생활용품,16,"96,437.50"


### 5. 완료 주문과 전체 주문 금액 차이
두 금액은 같은 상세의 수량×단가를 사용하고, 상태 포함 여부만 다르다. 차이가 취소·환불 상태 금액 합과 같은지도 확인했다.

In [15]:
excluded_amount = int(order_sales.loc[~order_sales.order_status.eq('completed'), 'line_total'].sum())
difference = all_order_amount - reference_total
assert difference == excluded_amount
display(pd.DataFrame({'범위': ['전체 주문', 'completed 주문', '차이 (= cancelled + refunded)'],
    '금액(원)': [all_order_amount, reference_total, difference]}))
print(f'전체 상태를 사용하면 completed 기준 대비 {difference / reference_total * 100:.2f}% 크게 계산된다.')

,범위,금액(원)
0,전체 주문,255610000
1,completed 주문,148990000
2,차이 (= cancelled + refunded),106620000


전체 상태를 사용하면 completed 기준 대비 71.56% 크게 계산된다.


### 6. 병합 검증을 포함한 고객별 구매 금액 요청 프롬프트

아래는 추가 실습 문항에 맞춰 작성한 프롬프트다. 별도 대화에 입력한 기록을 뜻하지 않는다. 실제 대응 코드는 본문 3~5번에서 실행했다.

```text
orders, order_items, customers CSV로 고객별 completed 주문 금액을 pandas로 계산해 줘.
기간은 2025-09-09부터 2026-09-09까지 양 끝 날짜를 포함해.
먼저 실제 컬럼과 order_status 값, 키의 결측·중복을 확인해.
order_items의 quantity * unit_price를 line_total로 만들어.
orders.order_id 고유성을 확인하고 order_items와 orders를
order_id 기준 left merge하되 validate='many_to_one', indicator=True를 사용해.
병합 전후 행 수, 미연결, 날짜 변환 실패가 있으면 조용히 제외하지 말고 알려줘.
completed와 지정 기간만 선택하고 customer_id로 합계 금액과
order_id.nunique() 기준 주문 수를 집계해.
customers에서는 customer_id와 city만 one_to_one으로 연결하고 다시 검증해.
이름은 출력하지 말고 Customer ID 라벨을 사용해.
고객별 total_sales 합과 원본 completed 상세 금액을 대조해.
결과를 CSV로 저장하고 다시 읽어서 값이 유지되는지 확인해.
금액을 회계상 순매출이나 이익이라고 단정하지 마.
```

### 추가 실습 결과의 해석
40세 이상은 78명이다. 최저 가격은 전자기기 상품 006과 뷰티 상품 028의 5,000원이다. 결제수단은 kakao_pay 79건(26.33%), naver_pay 77건(25.67%), bank_transfer 74건(24.67%), card 70건(23.33%)이다. 등록 상품 평균 가격은 식품 137,142.86원이 가장 높고 생활용품 96,437.50원이 가장 낮았다.

나이 필터는 고객 특성 선택, 상품 가격 정렬·평균은 상품 목록, 결제수단 비율은 전체 주문이 기준이다. 서로 분모가 다르므로 이 결과들을 완료 주문 카테고리 순위의 직접 원인으로 설명하지 않았다.

## 최종 제출 체크

- [x] 가상환경에서 4개 CSV를 읽고 핵심 셀의 Output을 저장했다.
- [x] 실제 컬럼·dtype·값의 종류와 결측·중복·키 관계를 확인했다.
- [x] 날짜·completed 범위, line_total 계산식, 주문 수 기준을 명시했다.
- [x] orders·products 병합의 validate·indicator·행 수를 확인했다.
- [x] 카테고리·상품·월·고객 집계와 원본의 총액을 대조했다.
- [x] 요약 CSV 4개를 저장하고 모두 다시 읽어 비교했다.
- [x] LLM 예시를 검증하고 사용·수정·보류 판단을 구분했다.
- [x] 결과 관찰, 해석과 판단, 업무 의미, 한계를 구분했다.
- [x] 공식 Notebook의 추가 실습 6문항에 답했다.
- [x] 고객 이름을 출력하지 않았고 Secret을 포함하지 않았다.
- [x] 핵심 Evidence 6장이 GitHub에서 정상 표시되는지 확인했다.
- [x] 최종 Notebook 파일 URL을 eTL에 제출했다. (2026-09-30 오후 5:16, 제출됨 확인)